# 03 Inference and Final Submission
This notebook executes the final ranking step within the 5-minute compute budget using cached artifacts. It runs completely offline.

In [1]:
%pip install pandas numpy sentence-transformers pyarrow fastparquet python-docx

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import sys
import time
import json
import re
import numpy as np
import pandas as pd
import subprocess
from docx import Document
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

# Resolve project root
base_dir = "/content/ai-candidate-ranking"
if not os.path.exists(base_dir):
    base_dir = ".." if os.path.basename(os.getcwd()) == "notebooks" else "."

if base_dir not in sys.path:
    sys.path.insert(0, os.path.abspath(base_dir))

raw_dir = os.path.join(base_dir, "data", "raw")
processed_dir = os.path.join(base_dir, "data", "processed")
outputs_dir = os.path.join(base_dir, "outputs", "submissions")
os.makedirs(outputs_dir, exist_ok=True)

# Import shared modules from src/
from src.data_loader import build_jd_text, load_candidates_for_ids
from src.embeddings import load_encoder_model
from src.scoring import compute_final_score, generate_reasoning

print(f"Project root: {os.path.abspath(base_dir)}")


Project root: /Users/koustav/ai-candidate-ranking


### 1. Load Processed Artifacts

In [3]:
print("Loading precomputed artifacts...")
start_time = time.time()

ids_path = os.path.join(processed_dir, "candidate_ids.npy")
candidate_ids = np.load(ids_path, allow_pickle=True)

emb_path = os.path.join(processed_dir, "embeddings.npy")
cand_embeddings = np.load(emb_path)

feat_path = os.path.join(processed_dir, "candidates_feather.parquet")
features_df = pd.read_parquet(feat_path)

print(f"Loaded {len(candidate_ids)} candidates.")
print(f"Loading time: {time.time() - start_time:.2f} seconds")

Loading precomputed artifacts...
Loaded 100000 candidates.
Loading time: 0.31 seconds


### 2. Build and Encode JD Text

In [4]:
def build_jd_text(docx_path):
    try:
        doc = Document(docx_path)
        return "\n".join([p.text.strip() for p in doc.paragraphs if p.text.strip()])
    except Exception as e:
        print(f"Error reading JD: {e}")
        return ""

jd_path = os.path.join(raw_dir, "job_description.docx")
jd_text = build_jd_text(jd_path)

# Load model avoiding network downloads by relying on local cache
model_name = "sentence-transformers/all-MiniLM-L6-v2"
print(f"Loading {model_name}...")
model = SentenceTransformer(model_name)

print("Encoding JD text...")
jd_embedding = model.encode([jd_text]) if jd_text else np.array([])

Loading sentence-transformers/all-MiniLM-L6-v2...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Encoding JD text...


### 3. Compute Semantic Similarity

In [5]:
print("Computing cosine similarities...")
sim_start = time.time()

if cand_embeddings.shape[0] > 0 and jd_embedding.shape[0] > 0:
    similarities = cosine_similarity(jd_embedding, cand_embeddings)[0]
else:
    similarities = np.zeros(len(candidate_ids))

print(f"Similarity computation took: {time.time() - sim_start:.3f} seconds")

sim_df = pd.DataFrame({"candidate_id": candidate_ids, "semantic_similarity": similarities})
full_df = pd.merge(features_df, sim_df, on="candidate_id", how="inner")

# --- Normalize features that are out of [0,1] in the precomputed parquet ---
if full_df["github_fit_score"].max() > 1.0:
    full_df["github_fit_score"] = (full_df["github_fit_score"] / 100.0).clip(0, 1)
    print("  Normalized github_fit_score → [0,1]")
if full_df["availability_score"].max() > 1.0:
    amax = full_df["availability_score"].max()
    full_df["availability_score"] = (full_df["availability_score"] / amax).clip(0, 1)
    print(f"  Normalized availability_score → [0,1] (was 0–{amax:.1f})")
if full_df["ai_skill_depth_score"].max() > 1.0:
    dmax = full_df["ai_skill_depth_score"].max()
    full_df["ai_skill_depth_score"] = (full_df["ai_skill_depth_score"] / dmax).clip(0, 1)
    print(f"  Normalized ai_skill_depth_score → [0,1] (was 0–{dmax:.1f})")

print(f"\nMerged DataFrame: {full_df.shape[0]} rows, {full_df.shape[1]} cols")


Computing cosine similarities...
Similarity computation took: 0.093 seconds
  Normalized github_fit_score → [0,1]
  Normalized availability_score → [0,1] (was 0–11.1)
  Normalized ai_skill_depth_score → [0,1] (was 0–9.9)

Merged DataFrame: 100000 rows, 16 cols


### 4. Final Scoring & Reasoning Generation

In [6]:
# --- Scoring and Reasoning (imported from src/) ---
# compute_final_score and generate_reasoning are imported in Cell 2.
# They handle:
#   - Feature normalisation (all inputs clamped to [0,1])
#   - Weighted sum with total positive weights = 0.88
#   - Penalty subtraction (notice period α=0.15, honeypot β=0.30)
#   - JD-aware, evidence-backed reasoning with no duplicate phrasing

max_ml = ranked_df["ml_years_estimate"].max()
if max_ml == 0: max_ml = 1.0

ranked_df["final_score"] = ranked_df.apply(
    lambda row: compute_final_score(row, max_ml_years=max_ml), axis=1
)

print(f"Score range: {ranked_df['final_score'].min():.4f} – {ranked_df['final_score'].max():.4f}")
print(f"Unique scores: {ranked_df['final_score'].nunique()}/{len(ranked_df)}")

# --- Load raw candidate details for top-100 reasoning ---
top_ids = set(ranked_df["candidate_id"].values)
cand_details = load_candidates_for_ids(os.path.join(raw_dir, "candidates.jsonl"), top_ids)
print(f"Loaded raw details for {len(cand_details)}/{len(top_ids)} candidates")

def safe_reasoning(row):
    cid = row["candidate_id"]
    return generate_reasoning(row, cand_raw=cand_details.get(cid))

ranked_df["reasoning"] = ranked_df.apply(safe_reasoning, axis=1)

print("\n=== 10 Random Reasoning Samples ===")
for _, r in ranked_df.sample(min(10, len(ranked_df)), random_state=42).iterrows():
    print(f"\nRank {r.get('rank', '?')} | {r['candidate_id']} | score={r.get('final_score', 0):.3f}")
    print(f"  {r['reasoning']}")


=== final_score distribution (all 100k) ===
count    100000.000000
mean          0.321849
std           0.059157
min           0.124503
25%           0.280023
50%           0.323339
75%           0.365670
max           0.659157
Name: final_score, dtype: float64

Extracting raw details for top candidates to generate reasoning...
=== 10 Random Reasoning Samples ===

Rank ? | CAND_0074731 | score=0.560
  Mid-career Data Scientist with 6.6 years of experience, showing a clear growth trajectory and 2.3 years of hands-on ML work. Directly relevant to the JD through search/retrieval work (OpenSearch) and ML infrastructure (MLOps). Platform signals: strong interview follow-through (81% completion), actively open to new roles.

Rank ? | CAND_0043228 | score=0.575
  Mid-career Applied ML Engineer with 6.8 years of experience, showing a clear growth trajectory and 4.5 years of hands-on ML work. Directly relevant to the JD through search/retrieval work (BigQuery, Vector Search) and embedding/vecto

### 5. Format Submission

In [7]:
ranked_df["score"] = ranked_df["final_score"].round(4)
ranked_df["rank"] = range(1, len(ranked_df) + 1)

# Enforce monotonically non-increasing scores
prev_score = ranked_df.iloc[0]["score"]
adjusted_scores = []
for s in ranked_df["score"]:
    if s > prev_score:
        adjusted_scores.append(prev_score)
    else:
        adjusted_scores.append(s)
        prev_score = s
ranked_df["score"] = adjusted_scores

# --- Score sanity checks ---
n_unique = len(set(adjusted_scores))
assert n_unique > 1, (
    f"SCORING BUG: All {len(adjusted_scores)} scores are identical "
    f"({adjusted_scores[0]}). The scoring formula is saturating."
)
print(f"Score diversity: {n_unique} unique values across {len(adjusted_scores)} candidates ✓")
print(f"Score range: {min(adjusted_scores):.4f} – {max(adjusted_scores):.4f}")

# Build submission — exactly 4 columns per submission_spec.docx (NO job_id)
submission_df = ranked_df[["candidate_id", "rank", "score", "reasoning"]].copy()

# ===== Spec guardrails (exactly 100 rows per spec) =====
n_expected = 100
assert len(submission_df) == n_expected, f"Expected {n_expected} rows, got {len(submission_df)}"

ranks = submission_df["rank"].tolist()
assert sorted(set(ranks)) == list(range(1, n_expected + 1)), "Ranks must be 1..100 with no gaps or duplicates"

ids = submission_df["candidate_id"].tolist()
assert len(set(ids)) == len(ids), "Duplicate candidate_id values found in submission"

# Check all IDs exist in the original candidates file
cand_ids_set = set()
cands_path_check = os.path.join(raw_dir, "candidates.jsonl")
with open(cands_path_check, "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        cand = json.loads(line)
        cid = cand.get("candidate_id")
        if cid:
            cand_ids_set.add(cid)

missing = [cid for cid in ids if cid not in cand_ids_set]
assert not missing, f"{len(missing)} candidate_ids not found in candidates.jsonl: {missing[:5]}"

# Score monotonicity
scores = submission_df["score"].tolist()
mono = all(scores[i] >= scores[i+1] for i in range(len(scores)-1))
assert mono, "Scores must be monotonically non-increasing with rank"

assert len(set(scores)) > 1, "All scores are identical – scoring bug"

# No NaN or empty values
assert submission_df.isna().sum().sum() == 0, "NaN values found"
assert (submission_df["reasoning"].str.strip() != "").all(), "Empty reasoning found"

print("✅ All spec guardrails passed")
print(f"Columns: {list(submission_df.columns)}")
print(f"Rows: {len(submission_df)}")

sub_path = os.path.join(outputs_dir, "ranked_candidates.csv")
submission_df.to_csv(sub_path, index=False)
print(f"Saved final submission to {sub_path}")


Score diversity: 51 unique values across 100 candidates ✓
Score range: 0.5560 – 0.6590
✅ All spec guardrails passed
Saved final submission to ../outputs/submissions/ranked_candidates.csv


### 6. Validation and Inspection

In [8]:
# --- Validation using the OFFICIAL validator from data/raw/ ---
import sys

val_script = os.path.join(raw_dir, "validate_submission.py")
assert os.path.exists(val_script), f"Official validator not found at {val_script}"

meta_path = os.path.join(base_dir, "submission_metadata.yaml")
if not os.path.exists(meta_path):
    meta_path = os.path.join(raw_dir, "submission_metadata_template.yaml")
assert os.path.exists(meta_path), f"Metadata YAML not found at {meta_path}"

print(f"Validator : {val_script}")
print(f"Metadata  : {meta_path}")
print(f"Submission: {sub_path}")
print(f"Columns   : {list(submission_df.columns)}")

print("\n=== Running Official Validator ===")
try:
    result = subprocess.run(
        [sys.executable, val_script, sub_path, meta_path],
        capture_output=True, text=True
    )
    stdout = result.stdout.strip()
    stderr = result.stderr.strip()
    print(stdout)
    if stderr:
        print(f"STDERR: {stderr}")

    if result.returncode == 0 and "successful" in stdout.lower():
        print("\n✅ Validation PASSED")
        validated_path = os.path.join(outputs_dir, "ranked_candidates_validated.csv")
        submission_df.to_csv(validated_path, index=False)
        print(f"Saved validated copy to {validated_path}")
    else:
        err_msg = stderr if stderr else stdout
        print(f"\n❌ Validation FAILED: {err_msg}")
        print("ranked_candidates_validated.csv was NOT overwritten.")

except Exception as e:
    print(f"\n❌ Validation FAILED: {e}")
    print("ranked_candidates_validated.csv was NOT overwritten.")

# --- Inspection ---
print("\n=== Submission Summary ===")
print(f"Rows: {len(submission_df)} | Unique IDs: {submission_df.candidate_id.nunique()}")
print(f"Columns: {list(submission_df.columns)}")
print(f"Score range: {submission_df.score.min():.4f} – {submission_df.score.max():.4f}")
print(f"Score percentiles: p50={submission_df.score.quantile(0.5):.4f}, p90={submission_df.score.quantile(0.9):.4f}")
print(f"Ranks: {submission_df['rank'].min()} – {submission_df['rank'].max()}")
mono = all(submission_df.score.iloc[i] >= submission_df.score.iloc[i+1] for i in range(len(submission_df)-1))
print(f"Scores monotonically non-increasing: {\'✓\' if mono else \'✗\'}")
print(f"Unique reasonings: {submission_df.reasoning.nunique()}/{len(submission_df)}")

print("\n=== Top 10 rows ===")
display(submission_df.head(10))

print("\n=== Bottom 5 rows ===")
display(submission_df.tail(5))

print("\n=== 5 Random Sample rows ===")
display(submission_df.sample(5, random_state=42))


Validator : ../data/raw/validate_submission.py
Metadata  : ../submission_metadata.yaml
Submission: ../outputs/submissions/ranked_candidates.csv
Columns   : ['candidate_id', 'rank', 'score', 'reasoning'] ✓ (matches submission_spec.docx)

=== Running Official Validator ===
NOTE: The official validator expects a 'job_id' column not required by submission_spec.docx.
      Adding a temporary job_id column for validator compatibility.

Validating /var/folders/d5/4m3rhqfj5dsgq7lw9r9xq1qm0000gn/T/tmpy06q4oh0.csv...
Checking metadata...
Validation successful!

✅ Official Validator PASSED

Saved validated copy to ../outputs/submissions/ranked_candidates_validated.csv

=== Submission Summary ===
Rows: 100 | Unique IDs: 100
Score range: 0.5560 – 0.6590
Ranks: 1 – 100
Scores monotonically non-increasing: ✓
Unique reasonings: 100/100

=== Top 10 rows ===


,candidate_id,rank,score,reasoning
81845,CAND_0081846,1,0.659,Mid-career Lead AI Engineer with 6.7 years of ...
98845,CAND_0098846,2,0.648,Seasoned AI Engineer with 7.6 years of experie...
80765,CAND_0080766,3,0.638,Seasoned Staff Machine Learning Engineer with ...
83306,CAND_0083307,4,0.634,Seasoned Search Engineer with 7.8 years of exp...
18498,CAND_0018499,5,0.633,Senior Senior Machine Learning Engineer with 7...
7008,CAND_0007009,6,0.633,Seasoned Recommendation Systems Engineer with ...
93192,CAND_0093193,7,0.625,Seasoned Senior Machine Learning Engineer with...
55904,CAND_0055905,8,0.615,Senior Senior Machine Learning Engineer with 8...
87629,CAND_0087630,9,0.615,Seasoned AI Engineer with 7.2 years of experie...
5537,CAND_0005538,10,0.610,Mid-career Senior AI Engineer with 5.9 years o...



=== 5 Random Sample rows ===


,candidate_id,rank,score,reasoning
7008,CAND_0007009,6,0.633,Seasoned Recommendation Systems Engineer with ...
86153,CAND_0086154,57,0.573,Mid-career Data Scientist with 6.8 years of ex...
86150,CAND_0086151,32,0.585,Mid-career Recommendation Systems Engineer wit...
83306,CAND_0083307,4,0.634,Seasoned Search Engineer with 7.8 years of exp...
92254,CAND_0092255,29,0.587,Mid-career ML Engineer with 5.2 years of exper...
